# Thailand Demand Dashboard

Demand view of EPPO Table 2.3-4 (petroleum product sales), built from `scripts/update_thailand.py` → `data/processed/thailand/thailand_eppo_sales.parquet`.

## Sections

1. **Setup** — load parquet, native vs canonical slices
2. **Headline** — sum of unified primaries (kbd), with Q1 2025 imputed months marked
3. **View A — native primaries** — REGULAR, PREMIUM, HSD, J.P., etc.
4. **View B — canonical** — Gasoline, Diesel, Jet, … (rollup of native primaries)
5. **Recent trends** — last 24 months + MoM/YoY table
6. **YoY growth** — trailing 12 vs prior 12
7. **Seasonality index** — monthly index, last 5 years
8. **Seasonality by year** — `analytics.seasonality_by_year_chart`
9. **Interactive deep-dive** — product picker with provisional shading
10. **EPPO vs JODI** — cross-source comparison (kbd)
11. **EPPO vs Kayrros (jet)** — official J.P. sales vs flight nowcaster (kbd)

## Conventions

- EPPO native unit is **barrels/day** (`bbl/d`); charts use **kbd** (value / 1000). JODI uses **KBD** (thousand barrels/day).

- Jan–Mar 2025 use the Q1 3-month average (`is_provisional=True`) — shown dashed.
- `TOTAL` / parent aggregates are not in the parquet; headline sums leaf primaries.

## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_thailand.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_thailand.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.products import (
    CANONICAL_AGGREGATE_LABELS,
    CANONICAL_KIND_LABEL,
    SUBCATEGORY_TO_PRODUCT_KIND,
)

PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "thailand" / "thailand_eppo_sales.parquet"

df = pd.read_parquet(PARQUET_PATH)
df["date"] = pd.to_datetime(df["date"])

demand = df[df["metric_type"] == "TOTDEMO"].copy()

from analytics.units import convert_series

# EPPO stores absolute bbl/d; JODI KBD is thousand bbl/day.
demand["value_kbd"] = convert_series(
    demand["value"], "bbl/d", "kbd", date=demand["date"]
)
PLOT_COL = "value_kbd"

# Native primaries in the DB (LSD is all-zero — keep for completeness, drop from charts)
NATIVE_PRODUCTS = sorted(
    demand.loc[demand["product_canonical"].notna(), "product_native"].unique(),
    key=lambda s: s.strip(),
)
MAJOR_PRODUCTS = [p for p in NATIVE_PRODUCTS if p.strip() != "LSD"]
DISPLAY = {p: p.strip() for p in NATIVE_PRODUCTS}

# Canonical rollup: sum native primaries sharing product_canonical
demand_canonical = (
    demand[demand["product_canonical"].notna()]
    .groupby(["date", "product_canonical", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
)
# product_canonical == product_map Sub-category (e.g. "Diesel", "Gasoline")
demand_canonical["kind"] = demand_canonical["product_canonical"].map(
    SUBCATEGORY_TO_PRODUCT_KIND
)
demand_canonical["panel"] = demand_canonical["kind"].map(CANONICAL_KIND_LABEL)

CANONICAL_PANELS = sorted(demand_canonical["panel"].dropna().unique())

print(f"Loaded: {len(df):,} rows  ({df['date'].min().date()} → {df['date'].max().date()})")
print(f"Demand: {len(demand):,} rows | native primaries: {MAJOR_PRODUCTS}")
print(f"Provisional months: {demand['is_provisional'].sum():,} rows")
print(f"Canonical panels: {CANONICAL_PANELS}")
# Sanity: HSD Dec 2024 EPPO vs typical scale
_hsd = demand[(demand.product_native==" HSD") & (demand.date=="2024-12-01")]
if len(_hsd):
    print(f"  HSD Dec-2024: {_hsd.value.iloc[0]:,.0f} bbl/d = {_hsd.value_kbd.iloc[0]:,.1f} kbd")

Loaded: 3,760 rows  (1986-01-01 → 2026-05-01)
Demand: 3,760 rows | native primaries: ['FUEL OIL', ' HSD', 'J.P.', 'KEROSENE', 'LPG', ' PREMIUM', ' REGULAR']
Provisional months: 24 rows
Canonical panels: ['Diesel', 'Fuel oil', 'Gasoline', 'Jet fuel', 'Kerosene', 'LPG']
  HSD Dec-2024: 442,174 bbl/d = 442.2 kbd


## 2. Headline — total sales (sum of primaries)

EPPO publishes a `TOTAL` row in the source files but we exclude it from the DB to avoid double-counting. The headline series sums all native primaries (excl. zero LSD).

In [2]:
headline = (
    demand[demand["product_native"].isin(MAJOR_PRODUCTS)]
    .groupby(["date", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
    .sort_values("date")
)

fig = go.Figure()
obs = headline[~headline["is_provisional"]]
prov = headline[headline["is_provisional"]]
fig.add_trace(go.Scatter(x=obs["date"], y=obs["value_kbd"], mode="lines", name="Observed"))
if not prov.empty:
    fig.add_trace(go.Scatter(
        x=prov["date"], y=prov["value_kbd"], mode="lines+markers",
        name="Q1 2025 imputed", line=dict(dash="dash"),
    ))
roll = headline.groupby("date", as_index=False)["value_kbd"].sum()["value_kbd"].rolling(12, min_periods=6).mean()
fig.add_trace(go.Scatter(
    x=headline["date"], y=roll, mode="lines",
    name="12-month rolling avg", line=dict(dash="dot", width=2),
))
fig.update_layout(
    title="Thailand petroleum sales — sum of primaries (kbd)",
    xaxis_title="Date", yaxis_title="Barrels / day (kbd)", height=440,
    hovermode="x unified", template="plotly_white",
)
fig.show()

## 3. View A — native primary products

In [3]:
mp = demand[demand["product_native"].isin(MAJOR_PRODUCTS)].copy()
mp["label"] = mp["product_native"].map(DISPLAY)

fig = px.line(
    mp, x="date", y="value_kbd", color="label",
    title="Sales by native product (kbd)",
    labels={"value_kbd": "kbd", "date": ""},
)
fig.update_layout(height=480, hovermode="x unified", template="plotly_white")
fig.show()

## 4. View B — canonical products

In [4]:
fig = px.line(
    demand_canonical, x="date", y="value_kbd", color="panel",
    title="Sales by canonical product (REGULAR + PREMIUM → Gasoline, etc.)",
    labels={"value_kbd": "kbd", "date": ""},
)
fig.update_layout(height=480, hovermode="x unified", template="plotly_white")
fig.show()

## 5. Recent trends (last 24 months)

In [5]:
cutoff = demand["date"].max() - pd.DateOffset(months=23)
recent = demand[
    (demand["date"] >= cutoff) & (demand["product_native"].isin(MAJOR_PRODUCTS))
].copy()
recent["label"] = recent["product_native"].map(DISPLAY)

fig = px.line(recent, x="date", y="value_kbd", color="label", title="Last 24 months (kbd)")
fig.update_layout(height=420, template="plotly_white")
fig.show()

tbl = recent.sort_values(["label", "date"]).copy()
tbl["mom_pct"] = tbl.groupby("label")["value_kbd"].pct_change(periods=1) * 100
tbl["yoy_pct"] = tbl.groupby("label")["value_kbd"].pct_change(periods=12) * 100
latest = tbl.groupby("label").tail(1)[["date", "value_kbd", "mom_pct", "yoy_pct", "is_provisional"]]
display(latest.sort_values("value_kbd", ascending=False).round(1))

,date,value_kbd,mom_pct,yoy_pct,is_provisional
3752,2026-05-01,407.1,11.3,-8.1,False
3759,2026-05-01,208.4,0.4,0.6,False
3754,2026-05-01,158.3,4.9,1.4,False
3757,2026-05-01,94.4,-12.6,-18.0,False
3755,2026-05-01,34.0,3.2,-19.1,False
3756,2026-05-01,32.6,7.0,-2.3,False
3758,2026-05-01,0.1,21.2,22.9,False


## 6. Year-over-year growth (trailing 12 vs prior 12)

In [6]:
mp_full = demand[demand["product_native"].isin(MAJOR_PRODUCTS)].copy()
mp_full["label"] = mp_full["product_native"].map(DISPLAY)
last_date = mp_full["date"].max()
window_end = last_date
window_start = last_date - pd.DateOffset(months=11)
prior_end = window_start - pd.DateOffset(months=1)
prior_start = prior_end - pd.DateOffset(months=11)

def _mean_in_range(g, start, end):
    sl = g[(g["date"] >= start) & (g["date"] <= end)]
    return sl["value_kbd"].mean() if len(sl) else np.nan

rows = []
for label, g in mp_full.groupby("label"):
    cur = _mean_in_range(g, window_start, window_end)
    prev = _mean_in_range(g, prior_start, prior_end)
    yoy = (cur / prev - 1) * 100 if prev and prev > 0 else np.nan
    rows.append({"product": label, "trailing_12_avg_kbd": cur, "prior_12_avg_kbd": prev, "yoy_pct": yoy})
yoy_df = pd.DataFrame(rows).sort_values("yoy_pct")

fig = px.bar(
    yoy_df, x="yoy_pct", y="product", orientation="h",
    title=f"YoY change in trailing-12mo avg kbd (to {last_date:%Y-%m})",
    labels={"yoy_pct": "YoY %"},
)
fig.update_layout(height=360, template="plotly_white")
fig.show()
display(yoy_df.round(1))

,product,trailing_12_avg_kbd,prior_12_avg_kbd,yoy_pct
6,REGULAR,39.3,43.7,-10.0
1,HSD,417.8,428.9,-2.6
3,KEROSENE,0.1,0.1,-2.0
4,LPG,212.5,214.0,-0.7
5,PREMIUM,161.1,154.3,4.4
2,J.P.,110.6,105.7,4.7
0,FUEL OIL,34.8,32.6,6.8


## 7. Seasonality index (last 5 complete years)

In [7]:
last_year = int(demand["date"].dt.year.max())
years = list(range(last_year - 5, last_year))
mp_idx = demand[demand["product_native"].isin(MAJOR_PRODUCTS)].copy()
mp_idx["label"] = mp_idx["product_native"].map(DISPLAY)
mp_idx["year"] = mp_idx["date"].dt.year
mp_idx["month"] = mp_idx["date"].dt.month
mp_idx = mp_idx[mp_idx["year"].isin(years)]

annual = mp_idx.groupby(["label", "year"])["value_kbd"].mean().rename("annual_mean")
monthly = mp_idx.groupby(["label", "year", "month"])["value_kbd"].mean().reset_index()
monthly = monthly.merge(annual, on=["label", "year"])
monthly["index"] = 100 * monthly["value_kbd"] / monthly["annual_mean"]
monthly["month_name"] = pd.to_datetime(monthly["month"], format="%m").dt.strftime("%b")

fig = px.line(
    monthly, x="month", y="index", color="label", facet_col="label",
    facet_col_wrap=2, title="Seasonality index (100 = annual mean, last 5 years)",
)
fig.update_xaxes(tickmode="linear", tick0=1, dtick=1)
fig.update_layout(height=900, template="plotly_white", showlegend=False)
fig.show()

## 8. Seasonality by calendar year

In [8]:
from analytics import seasonality_by_year_chart

season_df = demand[
    (demand["product_native"].isin(MAJOR_PRODUCTS)) & (~demand["is_provisional"])
].copy()

fig = seasonality_by_year_chart(
    season_df,
    products=MAJOR_PRODUCTS,
    product_col="product_native",
    title="Thailand - Seasonality by calendar year — native primaries (observed months only)",
    units_label="kbd",
)
fig.show()

## 9. Interactive product deep-dive

In [9]:
def plot_product_deep_dive(product_native: str) -> None:
    sl = demand[demand["product_native"] == product_native].sort_values("date")
    if sl.empty:
        print(f"No data for {product_native!r}")
        return
    title = DISPLAY.get(product_native, product_native.strip())
    fig = go.Figure()
    obs = sl[~sl["is_provisional"]]
    prov = sl[sl["is_provisional"]]
    fig.add_trace(go.Scatter(x=obs["date"], y=obs["value_kbd"], mode="lines", name="Observed"))
    if not prov.empty:
        fig.add_trace(go.Scatter(
            x=prov["date"], y=prov["value_kbd"], mode="lines+markers",
            name="Q1 2025 imputed", line=dict(dash="dash"),
        ))
    roll = sl["value_kbd"].rolling(12, min_periods=6).mean()
    fig.add_trace(go.Scatter(
        x=sl["date"], y=roll, mode="lines",
        name="12-month rolling avg", line=dict(dash="dot"),
    ))
    fig.update_layout(
        title=f"{title} — Thailand sales (kbd)", height=400,
        template="plotly_white", hovermode="x unified",
    )
    fig.show()


picker = widgets.Dropdown(
    options=[(DISPLAY[p], p) for p in MAJOR_PRODUCTS],
    description="Product",
)
widgets.interact(plot_product_deep_dive, product_native=picker)

interactive(children=(Dropdown(description='Product', options=(('FUEL OIL', 'FUEL OIL'), ('HSD', ' HSD'), ('J.…

<function __main__.plot_product_deep_dive(product_native: str) -> None>

## 10. EPPO vs JODI (TOTDEMO)

- **EPPO:** `value` is native `bbl/d` → `value_kbd` via `analytics.units`
- **JODI:** `unit_measure == "KBD"` only
- **Kerosene panel:** JODI `KEROSENE` is a parent aggregate; EPPO splits **J.P.** (jet) and **KEROSENE** — summed for this comparison


In [10]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run python scripts/update_jodi.py first.")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    jodi_agg = CANONICAL_AGGREGATE_LABELS["jodi"]
    eppo_agg = CANONICAL_AGGREGATE_LABELS["eppo_petroleum_sales"]
    kinds = sorted(set(eppo_agg) & set(jodi_agg))

    # JODI KEROSENE = JETKERO + X_OTHKERO (parent). EPPO reports jet as J.P. separately.
    eppo_xs = (
        demand[demand["product_canonical"].notna()]
        .assign(kind=lambda d: d["product_canonical"].map(SUBCATEGORY_TO_PRODUCT_KIND))
        .copy()
    )
    eppo_xs.loc[eppo_xs["kind"] == "jet", "kind"] = "kerosene"
    eppo_panel = eppo_xs.groupby(["date", "kind"], as_index=False)["value_kbd"].sum()
    eppo_panel["panel"] = eppo_panel["kind"].map(CANONICAL_KIND_LABEL)

    jodi_th = jodi[
        (jodi["ref_area"] == "TH")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
        & (jodi["energy_product"].isin(jodi_agg.values()))
    ].copy()
    jodi_kind_lookup = {v: k for k, v in jodi_agg.items()}
    jodi_th["kind"] = jodi_th["energy_product"].map(jodi_kind_lookup)
    jodi_th["panel"] = jodi_th["kind"].map(CANONICAL_KIND_LABEL)
    jodi_th["value_kbd"] = jodi_th["obs_value"]

    panels = [CANONICAL_KIND_LABEL[k] for k in kinds if k in CANONICAL_KIND_LABEL]
    fig = cross_source_comparison_chart(
        df_a=eppo_panel,
        df_b=jodi_th,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kbd",
        value_col_b="value_kbd",
        label_a="EPPO (Thailand)",
        label_b="JODI",
        title="Thailand TOTDEMO — EPPO vs JODI (kbd)",
        units_label="kbd",
    )
    fig.show()

    cutoff_24 = eppo_panel["date"].max() - pd.DateOffset(months=23)
    print("\nMean |gap| over last 24 months (kbd):")
    for panel in panels:
        e = eppo_panel.loc[eppo_panel["panel"] == panel].set_index("date")["value_kbd"]
        j = jodi_th.loc[jodi_th["panel"] == panel].set_index("date")["value_kbd"]
        merged = pd.concat([e, j], axis=1, keys=["eppo", "jodi"]).dropna().loc[lambda m: m.index >= cutoff_24]
        if merged.empty:
            continue
        gap = (merged["eppo"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:12s}  mean|gap| = {gap:>8,.1f} kbd  ({pct:5.1f}% of JODI level)")



Mean |gap| over last 24 months (kbd):
  Diesel        mean|gap| =      0.8 kbd  (  0.2% of JODI level)
  Fuel oil      mean|gap| =      0.3 kbd  (  1.0% of JODI level)
  Gasoline      mean|gap| =      0.3 kbd  (  0.2% of JODI level)
  Kerosene      mean|gap| =      1.2 kbd  (  1.1% of JODI level)
  LPG           mean|gap| =      1.0 kbd  (  0.5% of JODI level)


## 11. EPPO vs Kayrros nowcaster — Thailand jet fuel

Cross-source sanity check using the Kayrros flight-based nowcaster (`get_consumption` from `kayros/jet_fuel`). The nowcaster aggregates jet fuel **burned in flight** by aircraft departing Thailand; EPPO **J.P.** reports jet fuel **sold / delivered** at Thai airports (Table 2.3-4).

- **EPPO:** native `J.P.`, kbd; Q1 2025 imputed months shown **dashed** on the levels panel.
- **Kayrros:** `scope='Thailand'`, monthly `avg_kbd`, `drop_incomplete=True`.
- **Gap KPIs** use observed EPPO months only (exclude imputed Q1 2025).
- **Coverage:** EPPO from 1986; Kayrros from ~2019. Overlap window drives gap stats and parity scatter.


In [11]:
"""Section 11: EPPO (J.P.) vs Kayrros nowcaster — Thailand jet fuel."""
import os
import sys

from analytics import cross_source_gap_chart

KAYROS_ROOT = PROJECT_ROOT.parent / "kayros" / "jet_fuel"
DB_PATH = KAYROS_ROOT / "data" / "jet_fuel.duckdb"

if not DB_PATH.exists():
    print(f"[skip] Kayrros DB not found at {DB_PATH}")
    print("       Build/update kayros/jet_fuel/data/jet_fuel.duckdb first.")
else:
    if str(KAYROS_ROOT) not in sys.path:
        sys.path.insert(0, str(KAYROS_ROOT))
    os.environ.setdefault("JET_FUEL_DB_PATH", str(DB_PATH))
    from src.export import get_consumption  # noqa: E402

    jp = demand[demand["product_native"] == "J.P."].sort_values("date")
    eppo_obs = (
        jp[~jp["is_provisional"]]
        .loc[:, ["date", "value_kbd"]]
        .rename(columns={"value_kbd": "kbd"})
    )
    eppo_prov = (
        jp[jp["is_provisional"]]
        .loc[:, ["date", "value_kbd"]]
        .rename(columns={"value_kbd": "kbd"})
    )

    now_raw = get_consumption(
        scope_type="country",
        scope="Thailand",
        freq="monthly",
        metric="avg_kbd",
        drop_incomplete=True,
    )
    nowcaster = (
        now_raw.rename(columns={"period_start": "date", "value": "kbd"})
        .loc[:, ["date", "kbd"]]
        .sort_values("date")
        .reset_index(drop=True)
    )

    fig = cross_source_gap_chart(
        eppo_obs,
        nowcaster,
        label_a="EPPO (J.P.)",
        label_b="Kayrros",
        value_col_a="kbd",
        value_col_b="kbd",
        gap_direction="b_minus_a",
        units_label="kbd",
        title="Thailand jet fuel: EPPO vs Kayrros nowcaster",
        height=620,
    )
    if not eppo_prov.empty:
        fig.add_trace(
            go.Scatter(
                x=eppo_prov["date"],
                y=eppo_prov["kbd"],
                mode="lines+markers",
                name="EPPO Q1 2025 imputed",
                line=dict(color="#1f77b4", dash="dash", width=1.6),
                hovertemplate=(
                    "<b>EPPO (imputed)</b><br>%{x|%Y-%m}: %{y:,.2f} kbd<extra></extra>"
                ),
            ),
            row=1,
            col=1,
        )
    fig.show()

    overlap = (
        eppo_obs.merge(nowcaster, on="date", how="inner", suffixes=("_eppo", "_now"))
        .sort_values("date")
        .reset_index(drop=True)
    )
    overlap["gap_kbd"] = overlap["kbd_now"] - overlap["kbd_eppo"]
    overlap["gap_pct"] = overlap["gap_kbd"] / overlap["kbd_eppo"] * 100

    recent = overlap.tail(24).dropna(subset=["gap_pct"])
    print(
        f"Overlap window: {overlap['date'].min().date()} → "
        f"{overlap['date'].max().date()}  ({len(overlap)} months)"
    )
    print()
    print(f"Last {len(recent)} overlapping months — Kayrros vs EPPO (observed only):")
    print(
        f"  mean signed gap : {recent['gap_kbd'].mean():+7.2f} kbd  "
        f"({recent['gap_pct'].mean():+6.2f}% of EPPO)"
    )
    print(
        f"  mean |gap|      : {recent['gap_kbd'].abs().mean():7.2f} kbd  "
        f"({recent['gap_pct'].abs().mean():6.2f}% of EPPO)"
    )
    print(
        f"  max  |gap|      : {recent['gap_kbd'].abs().max():7.2f} kbd  "
        f"({recent['gap_pct'].abs().max():6.2f}% of EPPO)"
    )

    lo = float(min(overlap["kbd_eppo"].min(), overlap["kbd_now"].min())) * 0.95
    hi = float(max(overlap["kbd_eppo"].max(), overlap["kbd_now"].max())) * 1.05

    fig2 = go.Figure()
    fig2.add_trace(
        go.Scatter(
            x=overlap["kbd_eppo"],
            y=overlap["kbd_now"],
            mode="markers",
            marker=dict(
                size=8,
                color=overlap.index,
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title="month index<br>(early→late)"),
            ),
            customdata=overlap["date"].dt.strftime("%Y-%m"),
            hovertemplate=(
                "%{customdata}<br>EPPO: %{x:,.2f} kbd"
                "<br>Kayrros: %{y:,.2f} kbd<extra></extra>"
            ),
            name="Months",
        )
    )
    fig2.add_trace(
        go.Scatter(
            x=[lo, hi],
            y=[lo, hi],
            mode="lines",
            line=dict(color="grey", dash="dash"),
            name="45° (perfect agreement)",
            hoverinfo="skip",
        )
    )
    fig2.update_layout(
        title="Parity: Kayrros vs EPPO (Thailand jet, kbd)",
        template="plotly_white",
        height=520,
        xaxis=dict(title="EPPO (J.P.) kbd", range=[lo, hi]),
        yaxis=dict(
            title="Kayrros kbd",
            range=[lo, hi],
            scaleanchor="x",
            scaleratio=1,
        ),
    )
    fig2.show()


Overlap window: 2018-12-01 → 2026-05-01  (85 months)

Last 24 overlapping months — Kayrros vs EPPO (observed only):
  mean signed gap :  -12.48 kbd  (-11.64% of EPPO)
  mean |gap|      :   12.48 kbd  ( 11.64% of EPPO)
  max  |gap|      :   17.14 kbd  ( 15.27% of EPPO)
